# System Design — Multi-Agent (LangGraph)

A LangGraph `StateGraph` with five specialised agents orchestrated by a **Supervisor**:

```
User Requirement
      │
      ▼
  SUPERVISOR   ──────────────────────────────────────────┐
      │                                                  │
      ▼                                                  │
  RESEARCHER          (web search + RAG query)           │
      │                                                  │
      ▼                                                  │
  ARCHITECT           (produce design document)          │
      │                                                  │
      ▼                                                  │
   WRITER             (create folder + write design.md)  │
      │                                                  │
      ▼                                                  │
  LIBRARIAN           (ingest RAG + update knowledge     │
      │                graph in Oracle DB)               │
      └──────────────────────────────────────────────────┘
                            DONE
```

**Prerequisites:**
- `ollama serve` with `llama3.2` and `nomic-embed-text` pulled
- Oracle DB env vars: `ORACLE_USER`, `ORACLE_PASS`, `ORACLE_TLS_DNS`
- Node.js available (for `npx duckduckgo-mcp-server`)

## 1. Install dependencies (run once)

In [ ]:
# !pip install fastmcp langchain-ollama langchain-oracledb langchain-core langgraph oracledb pydantic

## 2. Oracle DB connection

In [1]:
import os
import oracledb

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)
print("Connected to Oracle DB")

Connected to Oracle DB


## 3. Embedding model + Vector stores

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_oracledb.vectorstores import DistanceStrategy

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

rag_store = OracleVS(
    client=connection,
    table_name="SYSTEM_DESIGN_VECTORS",
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

kg_store = OracleVS(
    client=connection,
    table_name="SYSTEM_DESIGN_GRAPH",
    embedding_function=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

print("Vector stores ready: SYSTEM_DESIGN_VECTORS, SYSTEM_DESIGN_GRAPH")

Vector stores ready: SYSTEM_DESIGN_VECTORS, SYSTEM_DESIGN_GRAPH


## 4. LLM

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

## 5. RAG helpers

In [4]:
from langchain_core.documents import Document


def rag_retrieve(query: str, k: int = 4) -> str:
    results = rag_store.similarity_search(query, k=k)
    if not results:
        return "No existing design found for this query."
    parts = []
    for i, doc in enumerate(results, 1):
        src = doc.metadata.get("source", "unknown")
        parts.append(f"[{i}] (source: {src})\n{doc.page_content.strip()}")
    return "\n\n".join(parts)


def rag_ingest(text: str, source: str) -> str:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    docs = [Document(page_content=c, metadata={"source": source}) for c in chunks]
    rag_store.add_documents(docs)
    return f"Ingested {len(docs)} chunks from '{source}' into SYSTEM_DESIGN_VECTORS"


def kg_add_component(name: str, component_type: str, description: str, relationships: str = "") -> str:
    content = f"Component: {name}\nType: {component_type}\nDescription: {description}\nRelationships: {relationships}"
    doc = Document(
        page_content=content,
        metadata={"name": name, "type": component_type, "relationships": relationships}
    )
    kg_store.add_documents([doc])
    return f"Knowledge graph: added component '{name}' ({component_type})"


def kg_query(query: str, k: int = 5) -> str:
    results = kg_store.similarity_search(query, k=k)
    if not results:
        return "No components found in the knowledge graph."
    return "\n---\n".join(doc.page_content.strip() for doc in results)


print("RAG helpers defined")

RAG helpers defined


## 6. MCP tool loader (mcp SDK >= 2.0)

In [5]:
from typing import Any, Optional, Literal
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters
from langchain_core.tools import StructuredTool
from pydantic import create_model


def mcp_tool_to_langchain(mcp_tool, session: ClientSession) -> StructuredTool:
    """Wrap an MCP tool as a LangChain StructuredTool (mcp SDK >= 2.0)."""
    properties = mcp_tool.input_schema.get("properties", {})
    required = set(mcp_tool.input_schema.get("required", []))
    fields = {}
    for name, schema in properties.items():
        t = schema.get("type")
        enum_vals = schema.get("enum")
        default = schema.get("default")
        is_required = name in required
        if enum_vals:
            annotation = Literal[tuple(enum_vals)]
        elif t == "integer":
            annotation = int
        elif t == "number":
            annotation = float
        elif t == "boolean":
            annotation = bool
        else:
            annotation = str
        if is_required:
            fields[name] = (annotation, ...)
        elif default is not None:
            fields[name] = (Optional[annotation], default)
        else:
            fields[name] = (Optional[annotation], None)

    ArgsModel = create_model(f"{mcp_tool.name}_args", **fields)

    async def call_tool(**kwargs: Any) -> str:
        clean_args = {k: v for k, v in kwargs.items() if v is not None}
        result = await session.call_tool(mcp_tool.name, arguments=clean_args)
        return "\n".join(block.text for block in result.content if hasattr(block, "text"))

    return StructuredTool.from_function(
        coroutine=call_tool,
        name=mcp_tool.name,
        description=mcp_tool.description or "",
        args_schema=ArgsModel,
    )


SYSTEM_DESIGN_DIR = "/Users/zainabfirdaus/git/learn/system-design"
MCP_SERVER_PATH   = "/Users/zainabfirdaus/git/learn/python3/mcps/mcp_filesystem_server.py"

fs_server_params = StdioServerParameters(command="python",  args=[MCP_SERVER_PATH])
ddg_server_params = StdioServerParameters(command="npx", args=["-y", "duckduckgo-mcp-server"])

print("MCP loader defined")

MCP loader defined


## 7. LangGraph state definition

In [6]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class DesignState(TypedDict):
    # Full message history shared across all agents
    messages: Annotated[list[BaseMessage], add_messages]
    # The original user requirement (set once, never changed)
    requirement: str
    # Research output from the Researcher agent
    research: str
    # Final design document text from the Architect agent
    design_doc: str
    # File path where the design was written
    design_path: str
    # Which agent should act next (set by Supervisor)
    next_agent: str


print("DesignState defined")

DesignState defined


## 8. Agent node builders

Each node is a `create_react_agent` wrapped in an async function that reads/writes `DesignState`.

In [7]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# ── Python tools (RAG + KG) ────────────────────────────────────────────────

@tool
def search_existing_designs(query: str) -> str:
    """Search SYSTEM_DESIGN_VECTORS in Oracle DB for existing system designs."""
    return rag_retrieve(query)

@tool
def query_knowledge_graph(query: str) -> str:
    """Search SYSTEM_DESIGN_GRAPH knowledge graph for existing components."""
    return kg_query(query)

@tool
def ingest_design_to_rag(text: str, source: str) -> str:
    """Ingest a system design document into SYSTEM_DESIGN_VECTORS Oracle DB."""
    return rag_ingest(text, source)

@tool
def add_component_to_knowledge_graph(name: str, component_type: str, description: str, relationships: str = "") -> str:
    """Add a system component node to the SYSTEM_DESIGN_GRAPH knowledge graph."""
    return kg_add_component(name, component_type, description, relationships)


# ── Node: RESEARCHER ───────────────────────────────────────────────────────

async def researcher_node(state: DesignState, fs_tools, ddg_tools) -> dict:
    """Searches RAG, knowledge graph, and web for prior art and best practices."""
    print("[RESEARCHER] Starting research...")

    agent = create_react_agent(
        model=llm,
        tools=ddg_tools + [search_existing_designs, query_knowledge_graph],
        prompt=(
            "You are a research specialist for system design. "
            "Given a requirement, you must:\n"
            "1. Search existing designs in Oracle DB using search_existing_designs.\n"
            "2. Query the knowledge graph using query_knowledge_graph.\n"
            "3. Search the web for latest architecture patterns and technologies.\n"
            "Summarise all findings clearly. Include relevant technologies, patterns, and trade-offs."
        )
    )

    result = await agent.ainvoke({
        "messages": [HumanMessage(content=f"Research the following requirement:\n{state['requirement']}")]
    })

    research_output = result["messages"][-1].content
    print(f"[RESEARCHER] Done. ({len(research_output)} chars)")

    return {
        "research": research_output,
        "messages": [AIMessage(content=research_output, name="researcher")],
        "next_agent": "architect",
    }


# ── Node: ARCHITECT ────────────────────────────────────────────────────────

async def architect_node(state: DesignState) -> dict:
    """Produces the full structured system design document."""
    print("[ARCHITECT] Producing design...")

    agent = create_react_agent(
        model=llm,
        tools=[],  # pure generation — no tools needed
        prompt=(
            f"You are a senior solution architect. Base file output directory: {SYSTEM_DESIGN_DIR}\n\n"
            "Given a requirement and research notes, produce a complete system design document in Markdown.\n"
            "The document MUST include these sections:\n"
            "# <Title>\n"
            "## Overview\n"
            "## Architecture Diagram (mermaid js)\n"
            "## Components\n"
            "## Data Flow\n"
            "## Scalability\n"
            "## Edge Cases\n"
            "## Technology Choices\n\n"
            "Also output a JSON block at the end listing major components:\n"
            "```json\n"
            '[{"name": "...", "type": "...", "description": "...", "relationships": "..."}]\n'
            "```"
        )
    )

    prompt = (
        f"Requirement:\n{state['requirement']}\n\n"
        f"Research findings:\n{state['research']}"
    )

    result = await agent.ainvoke({
        "messages": [HumanMessage(content=prompt)]
    })

    design_doc = result["messages"][-1].content
    print(f"[ARCHITECT] Done. ({len(design_doc)} chars)")

    return {
        "design_doc": design_doc,
        "messages": [AIMessage(content=design_doc, name="architect")],
        "next_agent": "writer",
    }


# ── Node: WRITER ───────────────────────────────────────────────────────────

async def writer_node(state: DesignState, fs_tools) -> dict:
    """Creates the folder structure and writes design.md to the filesystem."""
    print("[WRITER] Writing design to filesystem...")

    agent = create_react_agent(
        model=llm,
        tools=fs_tools,
        prompt=(
            f"You are a technical writer. Base directory: {SYSTEM_DESIGN_DIR}\n"
            "Given a design document and requirement, you must:\n"
            "1. Derive a short slug from the requirement (e.g. 'inventory-search').\n"
            f"2. Create directory: {SYSTEM_DESIGN_DIR}/<slug>/ using create_directory.\n"
            f"3. Write the full design document to: {SYSTEM_DESIGN_DIR}/<slug>/design.md using write_file.\n"
            "4. Confirm the file was written and return the exact file path."
        )
    )

    prompt = (
        f"Requirement:\n{state['requirement']}\n\n"
        f"Design document to write:\n{state['design_doc']}"
    )

    result = await agent.ainvoke({
        "messages": [HumanMessage(content=prompt)]
    })

    writer_output = result["messages"][-1].content

    # Extract the path from the output heuristically
    import re
    paths = re.findall(rf"{re.escape(SYSTEM_DESIGN_DIR)}/[\w\-]+/design\.md", writer_output)
    design_path = paths[0] if paths else f"{SYSTEM_DESIGN_DIR}/unknown/design.md"

    print(f"[WRITER] Written to: {design_path}")

    return {
        "design_path": design_path,
        "messages": [AIMessage(content=writer_output, name="writer")],
        "next_agent": "librarian",
    }


# ── Node: LIBRARIAN ────────────────────────────────────────────────────────

async def librarian_node(state: DesignState) -> dict:
    """Ingests the design into Oracle RAG and updates the knowledge graph."""
    print("[LIBRARIAN] Ingesting design into Oracle DB...")

    agent = create_react_agent(
        model=llm,
        tools=[ingest_design_to_rag, add_component_to_knowledge_graph],
        prompt=(
            "You are a knowledge librarian. Given a system design document, you must:\n"
            "1. Ingest the full design text using ingest_design_to_rag. "
            "Use the design file path (without extension) as the source identifier.\n"
            "2. Parse the JSON component list at the end of the design document.\n"
            "3. Call add_component_to_knowledge_graph for EACH component in the list.\n"
            "Confirm how many chunks were ingested and how many components were added."
        )
    )

    prompt = (
        f"Design file path: {state['design_path']}\n\n"
        f"Design document:\n{state['design_doc']}"
    )

    result = await agent.ainvoke({
        "messages": [HumanMessage(content=prompt)]
    })

    librarian_output = result["messages"][-1].content
    print(f"[LIBRARIAN] Done.")

    return {
        "messages": [AIMessage(content=librarian_output, name="librarian")],
        "next_agent": "DONE",
    }


print("Agent node functions defined")

Agent node functions defined


## 9. Supervisor node + LangGraph StateGraph

In [8]:
from langgraph.graph import StateGraph, END


def supervisor_router(state: DesignState) -> str:
    """Pure routing function — reads next_agent from state and returns the next node name."""
    next_agent = state.get("next_agent", "researcher")
    print(f"[SUPERVISOR] Routing to: {next_agent}")
    if next_agent == "DONE":
        return END
    return next_agent


def build_graph(fs_tools, ddg_tools):
    """Build and compile the multi-agent StateGraph."""

    # Bind MCP tools into node closures
    async def researcher(state):  return await researcher_node(state, fs_tools, ddg_tools)
    async def architect(state):   return await architect_node(state)
    async def writer(state):      return await writer_node(state, fs_tools)
    async def librarian(state):   return await librarian_node(state)

    graph = StateGraph(DesignState)

    # Add nodes
    graph.add_node("researcher", researcher)
    graph.add_node("architect",  architect)
    graph.add_node("writer",     writer)
    graph.add_node("librarian",  librarian)

    # Entry point
    graph.set_entry_point("researcher")

    # Each agent sets next_agent in state; supervisor_router reads it
    graph.add_conditional_edges("researcher", supervisor_router)
    graph.add_conditional_edges("architect",  supervisor_router)
    graph.add_conditional_edges("writer",     supervisor_router)
    graph.add_conditional_edges("librarian",  supervisor_router)

    return graph.compile()


print("Graph builder defined")

Graph builder defined


## 10. Run the multi-agent pipeline

In [9]:
async def run_multi_agent(requirement: str) -> DesignState:
    """Start both MCP servers, build the graph, and run the full pipeline."""
    async with stdio_client(fs_server_params) as (fs_read, fs_write):
        async with ClientSession(fs_read, fs_write) as fs_session:
            await fs_session.initialize()
            fs_response = await fs_session.list_tools()
            fs_tools = [mcp_tool_to_langchain(t, fs_session) for t in fs_response.tools]

            async with stdio_client(ddg_server_params) as (ddg_read, ddg_write):
                async with ClientSession(ddg_read, ddg_write) as ddg_session:
                    await ddg_session.initialize()
                    ddg_response = await ddg_session.list_tools()
                    ddg_tools = [mcp_tool_to_langchain(t, ddg_session) for t in ddg_response.tools]

                    print(f"FS tools  : {[t.name for t in fs_tools]}")
                    print(f"DDG tools : {[t.name for t in ddg_tools]}")

                    app = build_graph(fs_tools, ddg_tools)

                    initial_state: DesignState = {
                        "messages":    [HumanMessage(content=requirement)],
                        "requirement": requirement,
                        "research":    "",
                        "design_doc":  "",
                        "design_path": "",
                        "next_agent":  "researcher",
                    }

                    final_state = await app.ainvoke(initial_state)
                    return final_state


# ── Run ────────────────────────────────────────────────────────────────────
requirement = """
Design a system which can enable users to find a product in the inventory,
search for the product using natural language,
and fetch a particular model based on user query.
"""

final_state = await run_multi_agent(requirement)

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print(f"Design written to : {final_state['design_path']}")
print(f"\nFinal agent message:\n{final_state['messages'][-1].content}")

FS tools  : ['read_file', 'list_files', 'write_file', 'create_directory']
DDG tools : ['duckduckgo_web_search']
[RESEARCHER] Starting research...


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_95854/3899404426.py:35: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


[RESEARCHER] Done. (2738 chars)
[SUPERVISOR] Routing to: architect
[ARCHITECT] Producing design...


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_95854/3899404426.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


[ARCHITECT] Done. (4596 chars)
[SUPERVISOR] Routing to: writer
[WRITER] Writing design to filesystem...


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_95854/3899404426.py:115: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


[WRITER] Written to: /Users/zainabfirdaus/git/learn/system-design/inventory-search/design.md
[SUPERVISOR] Routing to: librarian
[LIBRARIAN] Ingesting design into Oracle DB...


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_95854/3899404426.py:159: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


[LIBRARIAN] Done.
[SUPERVISOR] Routing to: DONE

PIPELINE COMPLETE
Design written to : /Users/zainabfirdaus/git/learn/system-design/inventory-search/design.md

Final agent message:
{"name":"ingest_design_to_rag","parameters":{"text":"# Product Search System Design\n## Overview\n\nThe Product Search System is designed to enable users to find products in the inventory, search for products using natural language, and fetch specific product models based on user queries. The system utilizes Natural Language Processing (NLP) techniques, a search engine, and machine learning algorithms to achieve this functionality.\n\n## Architecture Diagram (Mermaid JS)\n```mermaid\ngraph LR\n    User Input[User Input] -->|Natural Language Processing (NLP)|> NLP Engine\n    NLP Engine -->|Search Results|> Search Engine\n    Search Engine -->|Indexed Data|> Product Inventory Management\n    Product Inventory Management -->|Product Data|> Machine Learning Model Retrieval\n    Machine Learning Model Retrieval 

## 11. Inspect the generated design document

In [10]:
print(final_state["design_doc"])

# Product Search System Design
## Overview

The Product Search System is designed to enable users to find products in the inventory, search for products using natural language, and fetch specific product models based on user queries. The system utilizes Natural Language Processing (NLP) techniques, a search engine, and machine learning algorithms to achieve this functionality.

## Architecture Diagram (Mermaid JS)
```mermaid
graph LR
    User Input[User Input] -->|Natural Language Processing (NLP)|> NLP Engine
    NLP Engine -->|Search Results|> Search Engine
    Search Engine -->|Indexed Data|> Product Inventory Management
    Product Inventory Management -->|Product Data|> Machine Learning Model Retrieval
    Machine Learning Model Retrieval -->|Retrieved Models|> User Output

    style User Input fill:#f9f,stroke:#333,stroke-width:3px
    style NLP Engine fill:#f9f,stroke:#333,stroke-width:3px
    style Search Engine fill:#f9f,stroke:#333,stroke-width:3px
    style Product Inventory

## 12. Inspect research output

In [11]:
print(final_state["research"])

Based on the search results, here is a summary of the findings:

**System Design:**

To design a system that enables users to find a product in the inventory, search for the product using natural language, and fetch a particular model based on user query, we can consider the following architecture:

1. **Natural Language Processing (NLP)**: Utilize NLP techniques such as tokenization, stemming, lemmatization, and entity recognition to preprocess user queries.
2. **Product Inventory Management**: Design a database schema to store product information, including product IDs, names, descriptions, prices, and inventory levels.
3. **Search Engine**: Implement a search engine that can index the product inventory data and support full-text searching using NLP techniques.
4. **Model Retrieval**: Use machine learning algorithms or knowledge graphs to retrieve specific product models based on user queries.

**Technologies:**

1. **Natural Language Processing (NLP)**: Utilize libraries such as NLT

## 13. Interactive — provide your own requirement

In [ ]:
custom_req = input("Enter your system design requirement: ").strip()
state = await run_multi_agent(custom_req)
print(f"\nDesign written to: {state['design_path']}")
print(state["design_doc"])

## 14. Inspect Oracle DB stores

In [12]:
print("=" * 60)
print("RAG search: 'inventory natural language search'")
print("=" * 60)
print(rag_retrieve("inventory natural language search", k=3))

print()
print("=" * 60)
print("Knowledge graph: 'vector search API gateway'")
print("=" * 60)
print(kg_query("vector search API gateway", k=5))

RAG search: 'inventory natural language search'
No existing design found for this query.

Knowledge graph: 'vector search API gateway'
No components found in the knowledge graph.
